In [1]:
import pymongo

def generate_index_page():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db["ciudad"]

    # Alle Städte holen, sortiert nach Provinz und Name
    cities = list(collection.find().sort([("Provincia", 1), ("Ciudad", 1)]))

    html_content = """
    <!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
    <html xmlns="http://www.w3.org/1999/xhtml">
    <head>
        <title>Mein Spanien-Archiv</title>
        <link rel="stylesheet" href="CSS/site.css" />
        <style>
            .provinc-block { margin-bottom: 30px; padding: 15px; border-left: 5px solid #2c3e50; background: #fdfdfd; }
            .city-link { margin-right: 10px; display: inline-block; padding: 5px 10px; background: #eee; border-radius: 3px; text-decoration: none; color: #333; }
            .city-link:hover { background: #34495e; color: white; }
        </style>
    </head>
    <body>
        <h1>🇪🇸 Spanien-Datenbank Index</h1>
        <p>Automatisch generiert aus MongoDB</p>
    """

    current_provinc = ""
    for city in cities:
        prov = city.get('Provincia', 'Unbekannt')
        if prov != current_provinc:
            if current_provinc != "": html_content += "</div>"
            html_content += f"<div class='provinc-block'><h2>Provinz: {prov}</h2>"
            current_provinc = prov
        
        # Link zur jeweiligen Unterseite (angenommen sie heißt Stadtname.html)
        name = city.get('Ciudad', 'Stadt')
        html_content += f"<a class='city-link' href='{name}.html'>{name}</a>"

    html_content += """
        </div>
        <p style='margin-top:50px;'><a href='#oben'>Nach oben</a></p>
    </body>
    </html>
    """

    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print("✅ Die index.html wurde frisch aus der MongoDB generiert!")

generate_index_page()

✅ Die index.html wurde frisch aus der MongoDB generiert!


In [2]:
import pymongo

def generate_audit_index():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db["ciudad"]

    cities = list(collection.find().sort([("Provincia", 1), ("Ciudad", 1)]))

    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Audit: Spanien-Datenbank</title>
        <style>
            body { font-family: sans-serif; background: #f4f7f6; padding: 20px; }
            .provinc-block { margin-bottom: 30px; background: white; padding: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); }
            .city-row { display: flex; justify-content: space-between; padding: 8px; border-bottom: 1px solid #eee; }
            .city-name { font-weight: bold; width: 250px; }
            .status-icons { font-size: 1.2em; }
            .missing { opacity: 0.2; filter: grayscale(1); } /* Blass, wenn Daten fehlen */
            h2 { color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        </style>
    </head>
    <body>
        <h1>📊 Daten-Audit: Sierra Suroeste & Co.</h1>
    """

    current_provinc = ""
    for city in cities:
        prov = city.get('Provincia', 'Unbekannt')
        if prov != current_provinc:
            if current_provinc != "": html_content += "</div>"
            html_content += f"<div class='provinc-block'><h2>Provinz: {prov}</h2>"
            current_provinc = prov
        
        # Check-Logik für die Icons
        has_coords = "📍" if city.get('Coordenadas') else "<span class='missing'>📍</span>"
        has_klima = "☀️" if city.get('Klima') else "<span class='missing'>☀️</span>"
        has_super = "🛒" if city.get('Supermercado') else "<span class='missing'>🛒</span>"
        has_biblio = "📚" if city.get('Biblioteca') else "<span class='missing'>📚</span>"
        
        name = city.get('Ciudad', 'Stadt')
        
        html_content += f"""
        <div class='city-row'>
            <span class='city-name'>{name}</span>
            <div class='status-icons'>
                {has_coords} {has_klima} {has_super} {has_biblio}
            </div>
        </div>
        """

    html_content += "</div></body></html>"

    with open("audit_index.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print("✅ Audit-Seite 'audit_index.html' wurde erstellt!")

generate_audit_index()

✅ Audit-Seite 'audit_index.html' wurde erstellt!


In [6]:
import pymongo

def get_all_fields(collection_name):
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    collection = db[collection_name]

    # Wir nutzen Aggregation, um alle Keys zu sammeln
    pipeline = [
        {"$project": {"arrayofkeyvalue": {"$objectToArray": "$$ROOT"}}},
        {"$unwind": "$arrayofkeyvalue"},
        {"$group": {"_id": None, "all_keys": {"$addToSet": "$arrayofkeyvalue.k"}}}
    ]

    result = list(collection.aggregate(pipeline))
    
    if result:
        fields = sorted(result[0]['all_keys'])
        print(f"--- Vorhandene Felder in {collection_name} ---")
        for field in fields:
            print(f"  - {field}")
    else:
        print("Keine Daten gefunden.")

get_all_fields("ciudad")

--- Vorhandene Felder in ciudad ---
  - 
  - 	
  - Aeropuerto
  - Alcalde
  - Alojamiento
  - Altitud
  - Autopista
  - Ayuntamiento
  - Ayuntamiento internet
  - Badminton
  - Baloncesto
  - Biblioteca
  - CN
  - Camping
  - Ciudad
  - Ciudad local
  - Com autónoma
  - Comarca
  - Coordenadas
  - Código postal
  - Dehesa
  - Densidad
  - Dentista
  - Economía
  - Empresa
  - Estacion
  - Estadio
  - Etimologia
  - Farmacia
  - Fútbol
  - Historia
  - Hospital
  - Iglesia
  - Isla
  - Juntas Generales
  - Klima
  - Librería
  - Lineas
  - Mancomunidad
  - Museo
  - Nautica
  - Núcleos de poplación
  - Otros
  - Pabellón
  - Parque
  - Parques
  - Partido judicial
  - Pedanías
  - Piscina
  - Playa
  - Población
  - Provincia
  - Puerto
  - Restaurante
  - Río
  - Superficie
  - Supermercado
  - Tennis
  - Tienda
  - Tienda de deportes
  - Tienda de ordenador
  - Timeline
  - Ubicación
  - Universidad
  - _id
  - curiosidades
  - embalse
  - texto


In [4]:
import pymongo

def cleanup_field_names():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    col = db["ciudad"]

    # Mapping: "Falscher Name": "Richtiger Name"
    corrections = {
        "Poplación": "Población",
        "Comaraca": "Comarca",
        "Supermercao": "Supermercado",
        "Histora": "Historia",
        "Pabellon": "Pabellón",
        "Libreria": "Librería",
        "Economia": "Economía",
        "Rio": "Río",
        "Tienda de computer": "Tienda de ordenador",
        "text": "texto"
    }

    for wrong, right in corrections.items():
        # Wir benennen nur um, wenn das alte Feld existiert
        result = col.update_many(
            {wrong: {"$exists": True}}, 
            {"$rename": {wrong: right}}
        )
        if result.modified_count > 0:
            print(f"✅ {wrong} -> {right}: {result.modified_count} Dokumente korrigiert.")

cleanup_field_names()

✅ Poplación -> Población: 656 Dokumente korrigiert.
✅ Comaraca -> Comarca: 1 Dokumente korrigiert.
✅ Supermercao -> Supermercado: 1 Dokumente korrigiert.
✅ Histora -> Historia: 1 Dokumente korrigiert.
✅ Pabellon -> Pabellón: 2 Dokumente korrigiert.
✅ Libreria -> Librería: 367 Dokumente korrigiert.
✅ Economia -> Economía: 12 Dokumente korrigiert.
✅ Rio -> Río: 419 Dokumente korrigiert.
✅ Tienda de computer -> Tienda de ordenador: 1 Dokumente korrigiert.
✅ text -> texto: 1 Dokumente korrigiert.


In [2]:
import pymongo

def audit_field_usage():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    col = db["ciudad"]
    
    total_docs = col.count_documents({})
    
    # Wir holen uns wieder alle Feldnamen
    pipeline = [
        {"$project": {"arrayofkeyvalue": {"$objectToArray": "$$ROOT"}}},
        {"$unwind": "$arrayofkeyvalue"},
        {"$group": {"_id": None, "all_keys": {"$addToSet": "$arrayofkeyvalue.k"}}}
    ]
    
    all_fields = list(col.aggregate(pipeline))[0]['all_keys']
    
    print(f"📊 Feld-Analyse (Gesamt-Dokumente: {total_docs})")
    print("-" * 50)
    
    report = []
    for field in sorted(all_fields):
        # Zähle Dokumente, in denen das Feld existiert UND nicht leer/null ist
        count = col.count_documents({field: {"$exists": True, "$ne": ""}})
        percentage = (count / total_docs) * 100
        report.append((field, count, percentage))
    
    # Sortiert nach Häufigkeit (seltenste zuerst)
    for field, count, pct in sorted(report, key=lambda x: x[2]):
        status = "⚠️ KAUM GENUTZT" if pct < 5 else "✅ AKTIV"
        print(f"{field:<25} | {count:>4} Einträge | {pct:>5.1f}% | {status}")

audit_field_usage()

📊 Feld-Analyse (Gesamt-Dokumente: 669)
--------------------------------------------------
	                         |    0 Einträge |   0.0% | ⚠️ KAUM GENUTZT
                          |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Dehesa                    |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Fiestas                   |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Igelesia                  |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Juntas Generales          |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Puerto                    |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Timeline                  |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
ayuntamiento_noticias     |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
curiosidades              |    1 Einträge |   0.1% | ⚠️ KAUM GENUTZT
Alcalde                   |    2 Einträge |   0.3% | ⚠️ KAUM GENUTZT
Baloncesto                |    2 Einträge |   0.3% | ⚠️ KAUM GENUTZT
Etimologia                |    2 Einträge |   0.3% | ⚠️ KAUM GENUTZT
Deportes     

In [1]:
import pymongo
from collections import defaultdict

def generate_provincial_top_facts():
    client = pymongo.MongoClient("mongodb://localhost:27017/")
    db = client["Espana"]
    col = db["ciudad"]

    # Sammle alle Städte mit relevanten Fakten
    pipeline = [
        {"$match": {
            "$or": [
                {"curiosidades": {"$exists": True, "$ne": ""}},
                {"Museo": {"$exists": True, "$ne": ""}},
                {"Otros": {"$exists": True, "$ne": ""}},
                {"Embalse": {"$exists": True, "$ne": ""}} # Optional, falls vorhanden
            ]
        }},
        {"$sort": {"Provincia": 1, "Ciudad": 1}} # Sortieren für die Ausgabe
    ]
    
    cities = list(col.aggregate(pipeline))
    provincial_data = defaultdict(list)
    
    # Gruppiere die Städte nach Provinz
    for city in cities:
        prov = city.get("Provincia", "Unbekannt")
        provincial_data[prov].append(city)

    # HTML-Generierung
    html_content = """
    <!DOCTYPE html>
    <html lang="de">
    <head>
        <meta charset="UTF-8">
        <title>Top-Sehenswürdigkeiten Spaniens pro Provinz</title>
        <style>
            body { font-family: sans-serif; background: #f8f9fa; padding: 30px; }
            h1 { text-align: center; color: #2c3e50; }
            .prov-block { background: white; margin-bottom: 25px; padding: 20px; border-radius: 8px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); }
            .prov-title { border-bottom: 2px solid #e74c3c; padding-bottom: 10px; color: #e74c3c; font-size: 1.8em; }
            .city-list { margin-top: 15px; }
            .city-entry { margin-bottom: 15px; padding: 10px; border-left: 4px solid #3498db; background: #fcfcfc; }
            .city-name { font-weight: bold; font-size: 1.2em; color: #34495e; }
            .fact-label { font-weight: bold; color: #7f8c8d; }
            .missing { color: #bdc3c7; font-style: italic; }
        </style>
    </head>
    <body>
        <h1>🇪🇸 Top-Sehenswürdigkeiten Spaniens</h1>
        <p style="text-align:center;">Automatisch generiert aus deinem Spanien-Archiv.</p>
    """

    for prov, cities_in_prov in sorted(provincial_data.items()):
        html_content += f"""
        <div class="prov-block">
            <h2 class="prov-title">Provinz: {prov} ({len(cities_in_prov)} Städte)</h2>
            <div class="city-list">
        """
        
        for city in cities_in_prov:
            name = city.get("Ciudad", "Stadt")
            
            # Wähle den besten verfügbaren Fakt
            museo = city.get("Museo", "").replace("<br>", " / ")
            curio = city.get("curiosidades", "").replace("<br>", " / ")
            otros = city.get("Otros", "").replace("<br>", " / ")
            embalse = city.get("Embalse", "").replace("<br>", " / ") # Falls vorhanden
            
            # Priorisiere die Anzeige (Museo > Curiosidades > Otros)
            top_fact = ""
            fact_source = ""
            
            if museo:
                top_fact = museo
                fact_source = "Museo / Highlight"
            elif curio:
                top_fact = curio
                fact_source = "Kurioses"
            elif otros:
                top_fact = otros
                fact_source = "Sonstiges"
            elif embalse:
                top_fact = embalse
                fact_source = "Stausee"
            
            if top_fact:
                html_content += f"""
                <div class="city-entry">
                    <div class="city-name">{name}</div>
                    <div><span class="fact-label">{fact_source}:</span> {top_fact}</div>
                </div>
                """
        
        html_content += """
            </div>
        </div>
        """

    html_content += "</body></html>"

    with open("top_highlights_index.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print("✅ Die Seite 'top_highlights_index.html' wurde frisch aus der MongoDB generiert!")

# Testlauf
generate_provincial_top_facts()

✅ Die Seite 'top_highlights_index.html' wurde frisch aus der MongoDB generiert!
